In [ ]:
# Import required libraries
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Data Loading

# Identify the Kaggle dataset
dataset = "avikdas567/longitudinal-egfr-trajectories-and-survival-in-ckd"

# Load the baseline CSV file
baseline_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    dataset,
    "baseline.csv"
)

# Load the followup CSV file
followup_df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    dataset,
    "followup.csv"
)

In [ ]:
# Display the first five rows of the baseline dataset
baseline_df.head()

In [15]:
# Display the first five rows of the followup dataset
followup_df.head()

,usubjid,randfl,ittfl,trtfl,anl01fl,aval,base,paramcd,param,ady,avisitn,avisit
0,id0001,Y,Y,Y,Y,54,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,1,0,BASELINE
1,id0001,Y,Y,Y,Y,51,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,22,3,WEEK 3
2,id0001,Y,Y,Y,Y,61,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,91,13,WEEK 13
3,id0001,Y,Y,Y,Y,63,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,183,26,WEEK 26
4,id0001,Y,Y,Y,Y,53,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,362,52,WEEK 52


In [ ]:
baseline_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   usubjid  5000 non-null   object
 1   trt01pn  5000 non-null   int64 
 2   randfl   5000 non-null   object
 3   ittfl    5000 non-null   object
 4   trtfl    5000 non-null   object
 5   blgfr    5000 non-null   int64 
 6   blglp1   5000 non-null   object
 7   strata   5000 non-null   object
dtypes: int64(2), object(6)
memory usage: 312.6+ KB


In [16]:
followup_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41800 entries, 0 to 41799
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   usubjid  41800 non-null  object
 1   randfl   41800 non-null  object
 2   ittfl    41800 non-null  object
 3   trtfl    41800 non-null  object
 4   anl01fl  41000 non-null  object
 5   aval     41800 non-null  int64 
 6   base     41800 non-null  int64 
 7   paramcd  41800 non-null  object
 8   param    41800 non-null  object
 9   ady      41800 non-null  int64 
 10  avisitn  41800 non-null  int64 
 11  avisit   41800 non-null  object
dtypes: int64(4), object(8)
memory usage: 3.8+ MB


In [ ]:
baseline_df.isnull().value_counts()

# Data Cleaning and Preprocessing

print('Checking for missing values in baseline dataset:')
print(baseline_df.isnull().value_counts())

print('\nChecking for missing values in followup dataset:')
print(followup_df.isnull().value_counts())

In [ ]:
for name, df in {'baseline_df': baseline_df, 'followup_df' : followup_df}.items():
    print(f'\n{name}')

    for column in df.select_dtypes(include=['object', 'string', 'category']).columns:
        if column == 'usubjid':
            continue
        print(f'\n{column}:')
        print(df[column].map(repr).value_counts(dropna=False).to_string())
    

Investigating the 800 unflagged rows: anl01fl (the analysis inclusion flag) is missing for 800 of 41,800 followup rows, and the dataset doesn't explain why. Here, I filter those rows and check which visits they belong to, and whether they duplicate a flagged visit for the same patient.

In [17]:
followup_df.head()

,usubjid,randfl,ittfl,trtfl,anl01fl,aval,base,paramcd,param,ady,avisitn,avisit
0,id0001,Y,Y,Y,Y,54,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,1,0,BASELINE
1,id0001,Y,Y,Y,Y,51,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,22,3,WEEK 3
2,id0001,Y,Y,Y,Y,61,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,91,13,WEEK 13
3,id0001,Y,Y,Y,Y,63,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,183,26,WEEK 26
4,id0001,Y,Y,Y,Y,53,54,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,362,52,WEEK 52


In [19]:
followup_df[followup_df['anl01fl'].isnull()]

,usubjid,randfl,ittfl,trtfl,anl01fl,aval,base,paramcd,param,ady,avisitn,avisit
32,id0004,Y,Y,Y,NaN,36,47,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,909,130,WEEK 130
92,id0012,Y,Y,Y,NaN,61,98,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,88,13,WEEK 13
141,id0017,Y,Y,Y,NaN,55,46,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,906,130,WEEK 130
282,id0035,Y,Y,Y,NaN,63,35,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,180,26,WEEK 26
439,id0054,Y,Y,Y,NaN,41,79,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,1091,156,WEEK 156
...,...,...,...,...,...,...,...,...,...,...,...,...
41530,id4968,Y,Y,Y,NaN,49,84,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,179,26,WEEK 26
41644,id4982,Y,Y,Y,NaN,60,32,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,362,52,WEEK 52
41696,id4988,Y,Y,Y,NaN,39,81,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,1091,156,WEEK 156
41702,id4989,Y,Y,Y,NaN,51,92,GFRBSCRT,GFR from Creatinine Adjusted for BSA (mL/min/1...,361,52,WEEK 52
